# Probe: K6 night-tail expert DQA-MoX

- created_utc: 2026-05-11T14:23:05+00:00
- target_mAP50: 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2`
- log: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2/logs/27c_probe_k6_night_tail_r2_train.log`
- research_note: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_note_iter_000_27c_probe_k6_night_tail_r2.md`

## Current Results

| trial | best mAP50 | mAP50:95 |
|---|---:|---:|
| 24a_client_dominant_soft_expand | 0.457000 | 0.258000 |
| 27a_soft_mixture_head_first_40_10 |  |  |
| 27b_localization_uncertainty_strict_then_open |  |  |
| 27b_probe_localization_uncertainty_r2 | 0.462000 | 0.260000 |

## Hypothesis

27b_probeはwarmup比+0.002 mAP50で止まり、改善はhighway_nightの+0.004程度に留まった。次は全体平均を少し触るのではなく、最悪splitのhighway_nightを明示的に踏むseedで、K=6の追加expertをnight/long-tailの受け皿にする。2 roundでwarmupを大きく超えないなら長いK6本走行はしない。

## Paper Basis

- FedMoX/PSSFL: https://arxiv.org/abs/2508.16568
  FedMoX treats the practical setting as server labeled high-resolution data plus client unlabeled low-resolution data, and uses sparse MoE with a spatial router and Soft-Mixture to stabilize semi-supervised FL.
- Rethinking Pseudo Labels: https://arxiv.org/abs/2106.00168
  Certainty-aware pseudo labels combine classification and localization quality, dynamically adjust thresholds, and reweight category losses to reduce class imbalance.
- CascadeMatch: https://machinelearning.apple.com/research/semi-supervised-long-tailed
  CascadeMatch uses progressive heads and data-driven pseudo-label mining for long-tailed SSOD; this is relevant because DQA-MoX repeatedly struggles with rare/night/client-specific slices.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2/logs/27c_probe_k6_night_tail_r2_train.log')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.6', '--num-experts', '6', '--top-k', '2', '--router-temperature', '1.4', '--router-balance-weight', '0.035', '--router-entropy-weight', '0.001', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.25', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--max-images-per-client', '0', '--master-port', '39000', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--warmup-epochs', '50', '--client-limit', '1200', '--client-sampling-ratio', '0.333', '--client-sampling-seed', '270206', '--phase1-rounds', '2', '--phase2-rounds', '0', '--phase1-train-scope', 'neck_head', '--phase1-repair-train-scope', 'neck_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00030', '--phase1-source-repeat', '1', '--phase1-pseudo-repeat', '3', '--phase1-loss-box', '0.00040', '--phase2-train-scope', 'all', '--phase2-repair-train-scope', 'all', '--phase2-client-epochs', '1', '--phase2-client-lr', '0.00004', '--phase2-source-repeat', '1', '--phase2-pseudo-repeat', '1', '--phase2-loss-box', '0.00004', '--server-repair-epochs', '1', '--server-repair-lr', '0.00018', '--server-repair-loss-box', '0.008', '--dqa-server-anchor', '0.14', '--dqa-min-server-alpha', '0.08', '--dqa-residual-blend', '0.00', '--late-dqa-server-anchor', '0.06', '--late-dqa-min-server-alpha', '0.02', '--late-dqa-residual-blend', '0.00', '--curriculum-start-round', '3', '--expert-keep-fraction', '0.86', '--expert-max-class-fraction', '0.38', '--actual-max-class-fraction', '0.56', '--late-expert-keep-fraction', '0.95', '--late-expert-max-class-fraction', '0.46', '--late-actual-max-class-fraction', '0.64', '--min-score', '0.16', '--min-stability', '0.50', '--late-min-score', '0.10', '--late-min-stability', '0.36', '--max-boxes-per-image', '14', '--skip-warmup-training', '--warmup-checkpoint', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


In [ ]:
import csv
from pathlib import Path

METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27c_probe_k6_night_tail_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)
